# Convolutional Neural Networks (CNNs)

Understanding CNNs for image classification.

## Learning Objectives

- Understand convolution operations
- Learn about pooling layers
- Build CNNs with PyTorch
- Train on image datasets
- Visualize learned features

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Understanding Convolution

A convolution slides a kernel (filter) over an image, computing element-wise products and sums.

In [ ]:
# Manual convolution example
def convolve2d(image, kernel):
    """Simple 2D convolution (no padding, stride=1)."""
    h, w = image.shape
    kh, kw = kernel.shape
    out_h, out_w = h - kh + 1, w - kw + 1
    
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

# Create sample image
image = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
], dtype=float)

# Edge detection kernels
kernels = {
    'Horizontal Edge': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]]),
    'Vertical Edge': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]),
    'Sharpen': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for i, (name, kernel) in enumerate(kernels.items()):
    output = convolve2d(image, kernel)
    axes[i+1].imshow(output, cmap='gray')
    axes[i+1].set_title(name)
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# PyTorch Conv2d
print("=== PyTorch Conv2d ===")

# Create a sample batch of images: (batch, channels, height, width)
x = torch.randn(1, 3, 32, 32)  # 1 RGB image, 32x32

# Conv2d layer
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)

output = conv(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nConv2d parameters:")
print(f"  - Weight shape: {conv.weight.shape}  # (out_channels, in_channels, kH, kW)")
print(f"  - Bias shape: {conv.bias.shape}")
print(f"  - Total params: {sum(p.numel() for p in conv.parameters())}")

## 2. Pooling Layers

Pooling reduces spatial dimensions while retaining important features.

In [ ]:
# Pooling operations
x = torch.randn(1, 1, 4, 4)
print(f"Input:\n{x[0, 0]}")

# Max pooling
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
max_output = max_pool(x)
print(f"\nMax Pool (2x2):\n{max_output[0, 0]}")

# Average pooling
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
avg_output = avg_pool(x)
print(f"\nAvg Pool (2x2):\n{avg_output[0, 0]}")

In [ ]:
# Visualize pooling
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

x_vis = torch.rand(1, 1, 8, 8)

axes[0].imshow(x_vis[0, 0].numpy(), cmap='viridis')
axes[0].set_title(f'Original (8x8)')

axes[1].imshow(nn.MaxPool2d(2)(x_vis)[0, 0].numpy(), cmap='viridis')
axes[1].set_title('MaxPool2d (4x4)')

axes[2].imshow(nn.AvgPool2d(2)(x_vis)[0, 0].numpy(), cmap='viridis')
axes[2].set_title('AvgPool2d (4x4)')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. CNN Architecture

In [ ]:
class SimpleCNN(nn.Module):
    """Simple CNN for MNIST classification."""
    
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 14x14 -> 14x14
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # 7x7 -> 7x7
        
        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, 10)
        
        # Dropout
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.conv1(x)))  # 28x28 -> 14x14
        
        # Conv block 2
        x = self.pool(F.relu(self.conv2(x)))  # 14x14 -> 7x7
        
        # Conv block 3
        x = self.pool(F.relu(self.conv3(x)))  # 7x7 -> 3x3
        
        # Flatten
        x = x.view(-1, 128 * 3 * 3)
        
        # Fully connected
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        
        return x

model = SimpleCNN().to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 4. Loading MNIST Dataset

In [ ]:
# Data transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Download MNIST
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for i in range(10):
    img, label = train_dataset[i]
    axes[i].imshow(img.squeeze(), cmap='gray')
    axes[i].set_title(f'Label: {label}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 5. Training the CNN

In [ ]:
def train_cnn(model, train_loader, test_loader, epochs=5, lr=0.001):
    """Train CNN model."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = output.max(1)
            train_total += target.size(0)
            train_correct += predicted.eq(target).sum().item()
            
            if batch_idx % 200 == 0:
                print(f'Epoch {epoch+1}/{epochs} [{batch_idx}/{len(train_loader)}] '
                      f'Loss: {loss.item():.4f}')
        
        # Evaluate
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, predicted = output.max(1)
                test_total += target.size(0)
                test_correct += predicted.eq(target).sum().item()
        
        train_acc = 100. * train_correct / train_total
        test_acc = 100. * test_correct / test_total
        
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        
        print(f'Epoch {epoch+1}: Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%')
    
    return history

# Train the model
model = SimpleCNN().to(device)
history = train_cnn(model, train_loader, test_loader, epochs=5)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['test_acc'], label='Test')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Visualizing Learned Features

In [ ]:
# Visualize first layer filters
filters = model.conv1.weight.data.cpu()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    if i < filters.shape[0]:
        ax.imshow(filters[i, 0], cmap='gray')
    ax.axis('off')

plt.suptitle('First Layer Filters (3x3)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize feature maps
def get_feature_maps(model, image):
    """Extract feature maps from each conv layer."""
    features = []
    x = image.unsqueeze(0).to(device)
    
    # Conv1
    x = F.relu(model.conv1(x))
    features.append(x.cpu().detach())
    x = model.pool(x)
    
    # Conv2
    x = F.relu(model.conv2(x))
    features.append(x.cpu().detach())
    x = model.pool(x)
    
    # Conv3
    x = F.relu(model.conv3(x))
    features.append(x.cpu().detach())
    
    return features

# Get a sample image
sample_img, sample_label = test_dataset[0]
feature_maps = get_feature_maps(model, sample_img)

# Plot
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(sample_img.squeeze(), cmap='gray')
axes[0].set_title(f'Original (Label: {sample_label})')
axes[0].axis('off')

for i, (fmap, name) in enumerate(zip(feature_maps, ['Conv1', 'Conv2', 'Conv3'])):
    # Show first 16 channels as a grid
    grid = fmap[0, :16].mean(dim=0)  # Average first 16 channels
    axes[i+1].imshow(grid, cmap='viridis')
    axes[i+1].set_title(f'{name} Features ({fmap.shape[2]}x{fmap.shape[3]})')
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

## 7. Making Predictions

In [ ]:
# Predict on test samples
model.eval()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.ravel()

for i in range(10):
    img, true_label = test_dataset[i]
    
    with torch.no_grad():
        output = model(img.unsqueeze(0).to(device))
        probs = F.softmax(output, dim=1)
        pred_label = output.argmax(1).item()
        confidence = probs[0, pred_label].item() * 100
    
    axes[i].imshow(img.squeeze(), cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    axes[i].set_title(f'Pred: {pred_label} ({confidence:.1f}%)\nTrue: {true_label}', color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 8. Key CNN Concepts

In [ ]:
import pandas as pd

concepts = pd.DataFrame({
    'Layer': ['Conv2d', 'MaxPool2d', 'BatchNorm2d', 'Dropout', 'Flatten'],
    'Purpose': [
        'Extract spatial features',
        'Reduce spatial dimensions',
        'Normalize activations',
        'Prevent overfitting',
        'Convert 2D to 1D'
    ],
    'Key Params': [
        'in_channels, out_channels, kernel_size',
        'kernel_size, stride',
        'num_features',
        'p (probability)',
        '-'
    ]
})
print(concepts.to_string(index=False))

## 9. Key Takeaways

1. **Convolutions** extract local features using learnable filters
2. **Pooling** reduces spatial dimensions and provides translation invariance
3. **Deeper layers** learn more abstract features
4. **Feature maps** show what the network "sees" at each layer
5. **Data augmentation** helps prevent overfitting
6. **Transfer learning** uses pre-trained CNNs for new tasks

### Common CNN Architectures
- LeNet-5 (1998): First successful CNN
- AlexNet (2012): Deep CNNs breakthrough
- VGG (2014): Very deep with 3x3 convs
- ResNet (2015): Skip connections
- EfficientNet (2019): Balanced scaling